In [0]:
# Imports
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import current_timestamp

print("Imports successful")

In [0]:
# Build DataFrame
from pyspark.sql.functions import date_format

country_data = [
    ("KE", "KEN", "Kenya",         "East Africa"),
    ("UG", "UGA", "Uganda",        "East Africa"),
    ("TZ", "TZA", "Tanzania",      "East Africa"),
    ("ET", "ETH", "Ethiopia",      "East Africa"),
    ("GH", "GHA", "Ghana",         "West Africa"),
    ("NG", "NGA", "Nigeria",       "West Africa"),
    ("ZA", "ZAF", "South Africa",  "Southern Africa"),
    ("SN", "SEN", "Senegal",       "West Africa"),
    ("ML", "MLI", "Mali",          "West Africa"),
    ("MZ", "MOZ", "Mozambique",    "Southern Africa"),
    ("RW", "RWA", "Rwanda",        "East Africa"),
    ("ZM", "ZMB", "Zambia",        "Southern Africa"),
    ("ZW", "ZWE", "Zimbabwe",      "Southern Africa"),
    ("CM", "CMR", "Cameroon",      "Central Africa"),
    ("CI", "CIV", "Cote d'Ivoire", "West Africa"),
]

schema = StructType([
    StructField("country_code", StringType(), False),
    StructField("country_iso3", StringType(), False),
    StructField("country_name", StringType(), False),
    StructField("region",       StringType(), False),
])

df_dim_country = spark.createDataFrame(country_data, schema=schema)

df_dim_country = df_dim_country.withColumn(
    "updated_at",
    date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
)

df_dim_country.show(15, truncate=False)

In [0]:
# Write to Unity Catalog

TABLE_NAME = "workspace.default.dim_country"

(df_dim_country.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_NAME)
)

print(f"✓ Written: {TABLE_NAME}")

In [0]:
# Verify
df_check = spark.table(TABLE_NAME)
print(f"Rows: {df_check.count()}")
display(df_check.orderBy("region", "country_name"))